# Supervisor Validation Demo

Loads pipeline outputs and prints summary statistics for quick validation.

## 1. Load Config

In [ ]:
import os
import sys

# Ensure the src package is importable when running from the notebooks directory
repo_root = os.path.abspath(os.path.join(os.path.dirname('__file__'), '..'))
src_path = os.path.join(repo_root, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from thesis_rx.config import load_config

config_path = os.path.join(repo_root, 'config', 'config.yaml')
cfg = load_config(config_path)
print('Config loaded.')
print(f"  output_dir : {cfg['project']['output_dir']}")
print(f"  run_exposure_sensitivity : {cfg['run']['run_exposure_sensitivity']}")

## 2. Determine Labels and Output Paths

In [ ]:
from pathlib import Path

output_dir = cfg['project']['output_dir']

labels = ['drug_era_primary']
if cfg['run']['run_exposure_sensitivity']:
    labels.append('drug_exposure_sensitivity')

SUFFIXES = [
    '_eligible.parquet',
    '_person_level_phenotypes.parquet',
    '_cluster_summary.parquet',
    '_top_ingredients.parquet',
]

def output_path(label, suffix):
    """Return the full parquet path for a given label and suffix."""
    return f"{output_dir}/{label}{suffix}"

# Check which paths are missing
missing_paths = []
for label in labels:
    for suffix in SUFFIXES:
        p = output_path(label, suffix)
        if not Path(p).exists():
            missing_paths.append(p)

if missing_paths:
    print('The following expected output paths are MISSING:')
    for p in missing_paths:
        print(f'  {p}')
    print()
    print('Generate them by running:')
    print('  spark-submit scripts/run_pipeline.py --config config/config.yaml')
else:
    print('All expected output paths found.')

## 3. Create Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, desc

spark = (
    SparkSession.builder
    .appName(cfg['project']['name'] + '-supervisor-demo')
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

## 4. Print Summary Statistics per Label

In [ ]:
for label in labels:
    print('=' * 60)
    print(f'LABEL: {label}')
    print('=' * 60)

    # --- eligible participants ---
    eligible_path = output_path(label, '_eligible.parquet')
    if Path(eligible_path).exists():
        eligible = spark.read.parquet(eligible_path)
        n_eligible = eligible.count()
        print(f"  Eligible participants : {n_eligible:,}")
    else:
        print(f"  [MISSING] {eligible_path}")

    # --- person-level phenotypes ---
    pheno_path = output_path(label, '_person_level_phenotypes.parquet')
    if Path(pheno_path).exists():
        final_person = spark.read.parquet(pheno_path)

        print()
        print('  Cluster counts (trajectory_cluster):')
        cluster_counts = (
            final_person
            .groupBy('trajectory_cluster')
            .count()
            .orderBy('trajectory_cluster')
        )
        cluster_counts.show(truncate=False)

        print('  Phenotype counts (discontinuation_phenotype):')
        phenotype_counts = (
            final_person
            .groupBy('discontinuation_phenotype')
            .count()
            .orderBy(desc('count'))
        )
        phenotype_counts.show(truncate=False)
    else:
        print(f"  [MISSING] {pheno_path}")

    # --- top ingredients per cluster ---
    top_path = output_path(label, '_top_ingredients.parquet')
    if Path(top_path).exists():
        top_ingredients = spark.read.parquet(top_path)
        print('  Top ingredients per cluster:')
        display_cols = ['rank', 'ingredient_concept_id']
        if 'concept_name' in top_ingredients.columns:
            display_cols.append('concept_name')
        display_cols.append('cluster_month_count')
        max_display_rows = cfg['run']['save_top_ingredients_per_cluster'] * len(top_ingredients.select('trajectory_cluster').distinct().collect())
        top_ingredients.select(*display_cols).orderBy('trajectory_cluster', 'rank').show(max_display_rows, truncate=False)
    else:
        print(f"  [MISSING] {top_path}")

    print()